# Notebook 6: Evaluation Metrics & Statistical Testing

Compute tracking metrics and statistical significance tests.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, get_ownship

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from defenses.defense_mechanisms import DefensePipeline, DefenseType

from defenses.defense_mechanisms import StatisticalSignificance

## 6.1 Load Data and Setup

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)

attacker = CameraAdversarialAttacker(epsilon=0.05)
attacked = {sid: df.copy() for sid, df in detections.items()}
attacked[3] = attacker.attack_detections(detections[3].copy(), AttackType.FGSM, sensor_id=3)

pipeline = DefensePipeline()
defended = pipeline.defend_detections(attacked, DefenseType.TEMPORAL_CONSISTENCY, ground_truth)

from helpers import compute_sensor_metrics
print('Setup complete')

## 6.2 Compute Per-Sensor Metrics

In [ ]:
conditions = {'Benign': detections, 'Attacked': attacked, 'Defended': defended}
all_metrics = {}

for condition_name, condition_dets in conditions.items():
    condition_metrics = {}
    for sensor_id in [1, 2, 3, 4]:
        m = compute_sensor_metrics(condition_dets[sensor_id], ground_truth, sensor_id)
        condition_metrics[sensor_id] = m
    all_metrics[condition_name] = condition_metrics

print('Detection Probability (%):')
print('Sensor     Benign     Attacked   Defended')
print('-' * 40)
for sensor_id in [1, 2, 3, 4]:
    b = all_metrics['Benign'][sensor_id]['detection_probability'] * 100
    a = all_metrics['Attacked'][sensor_id]['detection_probability'] * 100
    d = all_metrics['Defended'][sensor_id]['detection_probability'] * 100
    print(str(sensor_id) + '          ' + str(round(b, 1)) + '       ' + str(round(a, 1)) + '       ' + str(round(d, 1)))

## 6.3 Metrics Comparison Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sensors = [1, 2, 3, 4]
x = np.arange(len(sensors))
width = 0.25

benign_vals = [all_metrics['Benign'][s]['detection_probability'] * 100 for s in sensors]
attacked_vals = [all_metrics['Attacked'][s]['detection_probability'] * 100 for s in sensors]
defended_vals = [all_metrics['Defended'][s]['detection_probability'] * 100 for s in sensors]

ax.bar(x - width, benign_vals, width, label='Benign', color='blue', alpha=0.7)
ax.bar(x, attacked_vals, width, label='Attacked', color='red', alpha=0.7)
ax.bar(x + width, defended_vals, width, label='Defended', color='green', alpha=0.7)

ax.set_xlabel('Sensor ID')
ax.set_ylabel('Detection Probability (%)')
ax.set_title('Metrics: Benign vs Attacked vs Defended')
ax.set_xticks(x)
ax.set_xticklabels(sensors)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.4 Statistical Significance Testing

In [ ]:
stats = StatisticalSignificance()

def get_timestep_probs(dets, gt, sensor_id):
    probs = []
    times = sorted(dets[sensor_id]['time'].unique())
    for t in times[:50]:
        m = compute_sensor_metrics(dets[sensor_id], gt, sensor_id)
        probs.append(m['detection_probability'])
    return np.array(probs)

benign_probs = get_timestep_probs(detections, ground_truth, 3)
attacked_probs = get_timestep_probs(attacked, ground_truth, 3)
defended_probs = get_timestep_probs(defended, ground_truth, 3)

results = stats.compare_three_conditions(benign_probs, attacked_probs, defended_probs)

print('Statistical Significance Results (IR Camera):')
print('Comparison              Mean Diff    p-value      Significant  Cohen d')
print('-' * 75)
for comp, res in results.items():
    sig = 'YES ***' if res['significant'] else 'NO'
    print(comp + ' ' * (24 - len(comp)) + ' ' + str(round(res['mean_diff'], 4)) + '    ' + str(round(res['p_value'], 4)) + '    ' + sig + '    ' + str(round(res['cohens_d'], 3)))

## 6.5 Effect Size Visualization

In [ ]:
comparisons = list(results.keys())
cohens_d = [results[c]['cohens_d'] for c in comparisons]
p_values = [results[c]['p_value'] for c in comparisons]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['green' if d < 0.2 else 'yellow' if d < 0.5 else 'orange' if d < 0.8 else 'red' for d in cohens_d]
ax1.bar(comparisons, cohens_d, color=colors)
ax1.axhline(y=0.2, color='k', linestyle='--', alpha=0.5, label='Small')
ax1.axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='Medium')
ax1.axhline(y=0.8, color='k', linestyle='--', alpha=0.5, label='Large')
ax1.set_ylabel("Cohen's d")
ax1.set_title('Effect Size')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

ax2.bar(comparisons, p_values, color='steelblue')
ax2.axhline(y=0.05, color='r', linestyle='--', label='alpha = 0.05')
ax2.set_ylabel('p-value')
ax2.set_title('Statistical Significance')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Statistical Significance Testing Results', fontsize=14)
plt.tight_layout()
plt.show()

## 6.6 Per-Scenario Metrics Summary

In [ ]:
scenarios = ['scenario2', 'scenario3', 'scenario4']
scenario_metrics = []

for scen in scenarios:
    ld = load_scenario(scen)
    dets = get_all_detections(ld)
    gt = get_ground_truth(ld)
    
    att = CameraAdversarialAttacker(epsilon=0.05)
    att_dets = {sid: df.copy() for sid, df in dets.items()}
    att_dets[3] = att.attack_detections(dets[3].copy(), AttackType.FGSM, sensor_id=3)
    
    def_dets = pipeline.defend_detections(att_dets, DefenseType.TEMPORAL_CONSISTENCY, gt)
    
    m_benign = compute_sensor_metrics(dets[3], gt, 3)
    m_attacked = compute_sensor_metrics(att_dets[3], gt, 3)
    m_defended = compute_sensor_metrics(def_dets[3], gt, 3)
    
    scenario_metrics.append({
        'scenario': scen,
        'benign_detprob': m_benign['detection_probability'],
        'attacked_detprob': m_attacked['detection_probability'],
        'defended_detprob': m_defended['detection_probability'],
        'recovery': (m_defended['detection_probability'] / m_benign['detection_probability'] * 100) if m_benign['detection_probability'] > 0 else 0
    })

df_metrics = pd.DataFrame(scenario_metrics)
print(df_metrics.to_string(index=False))

## 6.7 Recovery Rate Across Scenarios

In [ ]:
plt.figure(figsize=(10, 6))

scenarios_names = [m['scenario'] for m in scenario_metrics]
benign_vals = [m['benign_detprob'] * 100 for m in scenario_metrics]
attacked_vals = [m['attacked_detprob'] * 100 for m in scenario_metrics]
defended_vals = [m['defended_detprob'] * 100 for m in scenario_metrics]

x = np.arange(len(scenarios_names))
width = 0.25

plt.bar(x - width, benign_vals, width, label='Benign', color='blue', alpha=0.7)
plt.bar(x, attacked_vals, width, label='Attacked', color='red', alpha=0.7)
plt.bar(x + width, defended_vals, width, label='Defended', color='green', alpha=0.7)

plt.xlabel('Scenario')
plt.ylabel('Detection Probability (%)')
plt.title('Defense Recovery Across Scenarios (IR Camera)')
plt.xticks(x, scenarios_names)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()